In [ ]:
import torch
from transformers import AutoTokenizer, MambaForCausalLM, AutoModel
from mamba_ssm.models.mixer_seq_simple import MambaLMHeadModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import pandas as pd
import os

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
data = pd.read_csv('data/disaster_tweets/train.csv')

In [4]:
data.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [5]:
data = data.sample(500)

In [6]:
data.shape

(485, 5)

In [ ]:
data[data["target"] == 1].shape[0], data[data["target"] == 0].shape[0]

In [7]:
strategy = "zero_shot"

In [12]:
model = MambaLMHeadModel.from_pretrained(f"havenhq/mamba-chat").to(device)
# model = AutoModel.from_pretrained(f"havenhq/mamba-chat", ignore_mismatched_sizes=True).to(device)
# model = MambaLMHeadModel.from_pretrained(os.path.expanduser("state-spaces/mamba-{model_size}"), device="cuda", dtype=torch.bfloat16)

d:\Capiro\Trabajos\tfm_ciencia_datos\mamba_evaluation\venv\lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


RuntimeError: Error(s) in loading state_dict for MambaModel:
	size mismatch for backbone.embeddings.weight: copying a param with shape torch.Size([50280, 2560]) from checkpoint, the shape in current model is torch.Size([50277, 768]).
	You may consider adding `ignore_mismatched_sizes=True` in the model `from_pretrained` method.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("havenhq/mamba-chat")

d:\Capiro\Trabajos\tfm_ciencia_datos\mamba_evaluation\venv\lib\site-packages\huggingface_hub\file_download.py:157: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Usuario\.cache\huggingface\hub\models--EleutherAI--gpt-neox-20b. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Special tokens have been added in the vocabulary, make sure the associated word 

In [ ]:
prompt_template_zero_shot = """
Instructions:

You have to analyze the following tweet and to determine if it speaks about a real desaster or not. Answer with "1" if the tweet speaks about a real disaster and with "0" if not. Don't add any other information in your answer.

--------------------------
Tweet:

{text}
--------------------------
Your answer (only a "1" or a "0"):
"""

In [ ]:
prompt_template_few_shot = """
Instructions:
Your task is to analyze the following tweet and determine if it is talking about a real disaster. A real disaster can include, but is not limited to, events such as earthquakes, hurricanes, fires, floods, major accidents, etc. If the tweet refers to a real disaster, respond with 1. If not, respond with 0.

Your response should only be the number 1 or 0.

Considerations:
Real Disasters: Significant events that impact people, property, or the environment.
Not Disasters: Personal opinions, jokes, fake news, or events that do not qualify as a disaster.

Examples:
Tweet: "A 7.5 magnitude earthquake has shaken the city, causing significant damage and injuries."
Expected Response: 1

Tweet: "I'm so tired that my house looks like a disaster after last night's party!"
Expected Response: 0

Tweet: "Uncontrolled wildfire in the north of the country. Evacuate immediately."
Expected Response: 1

Tweet: "It rained a lot yesterday, but today is sunny and beautiful."
Expected Response: 0

Tweet to Analyze:
Tweet: "{text}"

Response:
Result (1 or 0):
"""

In [ ]:
prompt_template = prompt_template_zero_shot if strategy == "zero_shot" else prompt_template_few_shot

In [ ]:
predictions = []
for index, row in data.iterrows():
    prompt = prompt_template.format(text=row['text'])
    messages = [dict(role="user", content=prompt)]
    input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to("cuda")
    out = model.generate(input_ids=input_ids, max_length=2000, temperature=0.9, top_p=0.7, eos_token_id=tokenizer.eos_token_id)
    decoded = tokenizer.batch_decode(out)
    try:
        predictions.append(int(decoded))
    except:
        print(f"{index}: {decoded}")
    if index % 50:
        print(index)

d:\Capiro\Trabajos\tfm_ciencia_datos\mamba_evaluation\venv\lib\site-packages\transformers\generation\configuration_utils.py:492: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
d:\Capiro\Trabajos\tfm_ciencia_datos\mamba_evaluation\venv\lib\site-packages\transformers\generation\configuration_utils.py:497: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
d:\Capiro\Trabajos\tfm_ciencia_datos\mamba_evaluation\venv\lib\site-packages\transformers\generation\configuration_utils.py:509: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `10` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warni


Instructions:

You have to analyze the following tweet and to determine if it speaks about a real desaster or not. Answer with "1" if the tweet speaks about a real disaster and with "0" if not. Don't add any other information in your answer.

--------------------------
Tweet:

Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all
--------------------------
Your answer (only a "1" or a "0"):

1. I am a man.
2. I am a man.
3. I am a man.
4. I am a man.
5. I am a man.
6. I am a man.
7. I am a man.
8. I am a man.
9. I am a man.
10. I am a man.
0: 
Instructions:

You have to analyze the following tweet and to determine if it speaks about a real desaster or not. Answer with "1" if the tweet speaks about a real disaster and with "0" if not. Don't add any other information in your answer.

--------------------------
Tweet:

Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all
--------------------------
Your answer (only a "1" or a "0"):

1. I am a man.
2. I am

KeyboardInterrupt: 

In [ ]:
predictions = []
with torch.no_grad():
    for index, row in data.iterrows():
        prompt = prompt_template.format(text=row['text'])
        encodings = tokenizer(prompt, return_tensors="pt")
        input_ids = encodings.to(device)
        #outputs = model(**input_ids, max_new_tokens=1)
        outputs = model(**input_ids)
        #p = tokenizer.decode(outputs.logits.argmax(dim=-1)[0], skip_special_tokens=True)
        p = tokenizer.decode(outputs.logits.argmax(dim=-1)[0])
        # predictions.append(p)
        try:
            predictions.append(int(p))
        except:
            print(f"{index}: {p}")
        if index % 50:
            print(index)

0: 
ructions


1 can to select the data data: then make if it is to the specific personerving. not.
 the ayes" if it tweet is about a real des and " "2" if it.
't use " other words. the tweet.

1

weet:

1 governmentbris
 not most Why our WarDesquake. 11 the helpget us..


 answer:1 if few1"): " "0"):


1: 
ructions


1 can to select the data data: then make if it is to the specific personerving. not.
 the ayes" if it tweet is about a real des and " "2" if it.
't use " other words. the tweet.

1

weet:

1ver of is the Jge,at




 tweet:1 one few1"): " "0"):


1


KeyboardInterrupt: 

In [ ]:
data["predictions"] = predictions

In [ ]:
f1_score(data["target"], data["predictions"])

0.0